In [ ]:
!pip install deepcut

In [ ]:
import deepcut
deepcut.tokenize('จะออกไปแตะขอบฟ้า แต่เหมือนว่าโชคชะตาไม่เข้าใจ')

In [ ]:
!pip install attacut

In [ ]:
from attacut import tokenize, Tokenizer

text = "จะออกไปแตะขอบฟ้า แต่เหมือนว่าโชคชะตาไม่เข้าใจ"

# tokenize using our best model `attacut-sc`
words = tokenize(text)

words

In [ ]:
!pip install OSKut
!pip install sefr_cut

In [ ]:
import oskut
oskut.load_model(engine='ws')
print(oskut.OSKut('จะออกไปแตะขอบฟ้า แต่เหมือนว่าโชคชะตาไม่เข้าใจ'))

In [ ]:
!pip install pythainlp

In [ ]:
from pythainlp.tokenize import word_tokenize
text = "จะออกไปแตะขอบฟ้า แต่เหมือนว่าโชคชะตาไม่เข้าใจ"

words=word_tokenize(text, engine="longest")
print(words)
print('')

In [ ]:
words=word_tokenize(text, engine="mm")
print(words)
print('')

In [ ]:
words=word_tokenize(text, engine="newmm")
print(words)
print('')

In [ ]:
words=word_tokenize(text, engine="newmm-safe")
print(words)
print('')

##part-of-speech tagging

In [ ]:
from pythainlp.tag import pos_tag, pos_tag_sents

text = "จะออกไปแตะขอบฟ้า แต่เหมือนว่าโชคชะตาไม่เข้าใจ"

words=word_tokenize(text, engine="longest")

pos_tag(words)

##example bg

In [ ]:
#ref: https://github.com/JagerV3/sentiment_analysis_thai

import pandas as pd
df=pd.read_csv('../../datasets/text_datasets/burgerking-UTF8-traindataset-3.csv')
df=df.rename(columns={'class':'class_label','message':'text'})
df

In [ ]:
import re
import string
from pythainlp.corpus.common import thai_stopwords
from pythainlp.tokenize import word_tokenize

thai_stopwords = list(thai_stopwords())
thai_stopwords

def perform_removal(word):
    #กำจัดช่องว่างก่อน/หลังคำ
    word = word.strip()

    #เปลี่ยนภาษาอังกฤษเป็นตัวอักษรตัวเล็ก
    word = word.lower()

    #กำจัดเครื่องหมายวรรคตอน
    word = word.translate(str.maketrans('','', string.punctuation))

    #กำจัด stop words และตัวเลขโดดๆ
    if(word.isdigit() ):
        return ""
    else:
        return word

def clean_text(text):
  text = "".join(u for u in text if u not in ("?", ".", ";", ":", "!", '"', "ๆ", "ฯ","'"))
  tokens=word_tokenize(text, engine="oskut", keep_whitespace=False)

  tokens = [word for word in tokens if word.lower not in thai_stopwords]

  tokens = [word for word in tokens if len(word)>1]

  tokens = [perform_removal(word) for word in tokens]


  text = ' '.join(tokens)
  return text

In [ ]:
df['text'] = df['text'].apply(lambda x:clean_text(x))

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

df_pos = df[df['class_label'] == 1]
pos_word_all = " ".join(text for text in df_pos['text'])
reg = r"[ก-๙a-zA-Z']+"
fp = 'THSarabunNew.ttf'
wordcloud = WordCloud(stopwords=thai_stopwords, background_color = 'white', max_words=2000, height = 2000, width=4000, font_path=fp, regexp=reg).generate(pos_word_all)
plt.figure(figsize = (16,8))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
df_neg = df[df['class_label'] == 0]
neg_word_all = " ".join(text for text in df_neg['text'])
wordcloud = WordCloud(stopwords=thai_stopwords, background_color = 'white', max_words=2000, height = 2000, width=4000, font_path=fp, regexp=reg).generate(neg_word_all)
plt.figure(figsize = (16,8))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.20, stratify=df['class_label'])

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

def identity_fun(text):
    return text.split()

'''
# CountVectorizer
vec = CountVectorizer(
    analyzer = 'word', #this is default
    tokenizer=identity_fun, #does no extra tokenizing
    ngram_range=(1,3),
    max_features=300
)
'''

#TFIDF
vec = TfidfVectorizer(
    analyzer = 'word', #this is default
    tokenizer=identity_fun, #does no extra tokenizing
    ngram_range=(1,3),
    max_features=300
)


vec.fit(df_train['text'])
vec.vocabulary_

In [ ]:
vec.get_feature_names_out()

In [ ]:
import numpy as np

#สุ่มช่วงของ 5 เอกสารที่ติดกันมาทดลองใช้งาน
count_vector= vec.transform(df_train['text'][:10])
count_array = np.array(count_vector.todense())

#แปลงเป็น DataFrame เพื่อง่ายแก่การอ่าน
df_x = pd.DataFrame(count_array,columns=vec.get_feature_names_out())
df_x

In [ ]:
X_train = vec.transform(df_train.text)
X_test = vec.transform(df_test.text)

y_train = df_train['class_label']
y_test = df_test['class_label']

In [ ]:
# Transform (TF-IDF)
'''
from sklearn.feature_selection import SelectKBest, mutual_info_classif

tfidf_train=vec.fit_transform(df_train.text)
tfidf_train = pd.DataFrame(np.array(tfidf_train.todense()),columns=vec.get_feature_names_out())

y_train = df_train['class_label']
y_test = df_test['class_label']

# Feature Selection
selector = SelectKBest(score_func=mutual_info_classif, k=200)
X_new = selector.fit_transform(tfidf_train, df_train.class_label)
X_train = pd.DataFrame(X_new,columns=selector.get_feature_names_out())

tfidf_test=vec.transform(df_test.text)
tfidf_test = pd.DataFrame(np.array(tfidf_test.todense()),columns=vec.get_feature_names_out())
X_test = selector.fit_transform(tfidf_test, df_test.class_label)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, classification_report

dt=DecisionTreeClassifier()
dt.fit(X_train, y_train)

preds = dt.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
from sklearn.tree import export_graphviz
import graphviz

# Export decision tree to graphviz format
dot_data = export_graphviz(dt, out_file=None,
                           feature_names = vec.get_feature_names_out(),
                           class_names=['Neg','Pos'],
                           filled=True, rounded=True,
                           special_characters=True)

# Visualize decision tree using graphviz
graph = graphviz.Source(dot_data)
graph.format = 'png'
graph.render("burger_decision_tree")
graph # Display decision tree